# Restore four reviewed partial fronts and replay the same predictions

**CPU only: run all four code cells.** This notebook compares the frozen
r50 evaluation reference with the separately versioned r67 reference.
Four justified partial-front rows are restored using exact historical
coordinates and corner-availability flags. Existing rows retain their
bytes and positions. Three uncertain candidates remain held.

The supplied original model, historical retained method and both arms
of PR157 and PR158 use their unchanged saved predictions. No photographs,
model weights, GPU, training or fresh neural inference are needed.
Standard-library analysis uses approximately 10.5 MB of archived inputs.

This measures a reference-population change, not learned improvement.
The same 20 photographs remain development evidence. The new reference
has 658 fronts and 583 complete corner pairs; partial-front detection is
reported separately. No ignore-region mechanism is introduced.


## 1 Load the fixed implementation and reference

Source and candidate labels come from an immutable commit. Existing
modified files are preserved and rejected; choose another
`BAOMAI_PARTIAL_INCLUSION_BASE` to run beside an edited checkout.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "2c3886271c12e501d236a5eff54e3eb78007d1b0"
SOURCE_HASHES = {'replay_partial_inclusion_r67.py': '5502022e8f5a7b510e875f2948d3dcbf4b1ab59addd94a9e5d3260ec3eda7f9a', 'analyze_duplicate_audit_r66.py': '868ab2c5df33ba376da4212aca6281d99a9d5c3a9d6cd1dac7dfaf4e1cd7acfa', 'evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'audit_reflection_return.py': '55f391535b040a27682e899aec74915b87864063023bc1ed818f8f82e67101de', 'predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6', 'run_corner_average_r58.py': '286232cf490f950b367068ea52b43078970ab27bb016169cb99159dab2fdbd88', 'run_original_comparison_r59.py': '279e2c0bd701299ea2a6f30403e97da68cc44a9c4ff874c5ee5a0d155a6e6ea3', 'run_reference_matrix_r60.py': '2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1', 'run_training_label_comparison_r61.py': '0dc8d5dfdd344f590f88d17ea7ced76f3fc018026fae221a4df6e25d7c807a9c', 'run_flip_inference_r56.py': 'bf2ff9056f12976ca8c4aa971de2d2a7af1c679369c44dbf73de9f9d2542700d', 'diagnose_fronts_r50.py': 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e', 'run_inference_ablation_r53.py': '8c8a37c3dcaccde5c568e9df6f44651cdb05965ca5f5ed5b0f2978852387dcc1'}
BASE = Path(os.environ.get("BAOMAI_PARTIAL_INCLUSION_BASE", "/content/baomai-partial-inclusion-r67")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
INPUT_DIR, OUTPUT = WORK / "inputs", WORK / "results"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing files preserved; choose a fresh BAOMAI_PARTIAL_INCLUSION_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip",
         "/data/reviews/fronts-2026-11-28-r67/", "/docs/thesis/evidence/partial-inclusion-r67/excluded-front-provenance.json"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Source changed; choose a fresh BAOMAI_PARTIAL_INCLUSION_BASE.")
    for name, expected in SOURCE_HASHES.items():
        if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Implementation hash differs: " + name)
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
import replay_partial_inclusion_r67 as audit
from IPython.display import Markdown, display
print("Immutable source:", SOURCE_COMMIT)


## 2 Verify saved inputs and append-only reference lineage

Verify archive sizes, hashes, membership, CRC and all member inventories.
Check each of the 20 candidate label files against archived r50 bytes,
the four accepted historical lines and the three explicit holds.
No historical reference or training-label file is written.


In [ ]:
audit.stage_inputs(SOURCE, INPUT_DIR)
spec = audit.INPUTS["historical"]
old = audit.verified_zip(INPUT_DIR / spec["filename"], spec["sha256"], spec["bytes"])
_, _, lineage, reference_manifest, _ = audit.candidate_reference(
    SOURCE, audit.read(old, "reference-labels.json.gz")["r50"])
print("Three immutable prediction archives verified.")
print("Candidate population:", reference_manifest["expected_counts"])
for row in reference_manifest["restored_rows"]:
    print(row["stable_id"], "->", row["candidate_reference_id"])
print("Held candidates:", [row["stable_id"] for row in reference_manifest["held_cases"]])


## 3 Replay baselines and measure reference-only changes

Archived predictions, view associations, scores and cases must reproduce
exactly. Each of the six r50 primary-method baselines is then checked
against its saved per-image scores before fresh matching against r67.

Keep all output records and coordinates unchanged. Report restored
targets, any retained-target reassignment, output-category transitions,
and detection with zero, one or two known corners alongside pair scores.

A separate eight-row supplement repeats the three previously fixed r66
output restrictions for the PR158 reviewed-label arm only. It checks
whether a filter removes an output now supported by a restored front.
It does not replace native NMS or choose a threshold.


In [ ]:
summary, records = audit.analyze(SOURCE, INPUT_DIR)
audit.write_outputs(OUTPUT, summary, records)
display(Markdown((OUTPUT / "results.md").read_text()))
print("Result rows:", len(summary["matrix"]))
print("Archived score objects replayed:", summary["provenance"]["replayed_image_scores"])
print("Reference-change ledgers:", summary["record_counts"])


## 4 Verify saved evidence and download

Verify every result member and package the matrix, full measurements,
per-image scores, exact before/after labels, target lineage, restored
outcomes and unchanged predictions. Source and input identities remain
explicit. No model or threshold is selected or promoted.


In [ ]:
verify_source()
RESULT = audit.package_results(OUTPUT, SOURCE_COMMIT)
manifest = {"source_commit": SOURCE_COMMIT, "notebook_code_cells": 4,
            "r50_baseline_replay": "exact", "fresh_inference": False,
            "training": False, "predictions_modified": False,
            "historical_references_modified": False, "training_labels_modified": False,
            "new_reference": reference_manifest["expected_counts"],
            "automatic_promotion": False,
            "result": {"filename": RESULT.name, "bytes": RESULT.stat().st_size,
                       "sha256": hashlib.sha256(RESULT.read_bytes()).hexdigest()}}
(WORK / "package-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))
